In [2]:
# Загрузка датасета с результатами АБ теста 
import pandas as pd

df = pd.read_csv('/mnt/HC_Volume_18315164/home-jupyter/jupyter-polina-manylova-bea7493/dating_data.csv')
df

,user_id_1,user_id_2,group,is_match
0,79,91,1,1
1,716,353,1,1
2,423,677,0,0
3,658,165,1,1
4,969,155,0,1
...,...,...,...,...
14509,476,631,0,0
14510,455,770,1,0
14511,260,6,1,1
14512,885,812,1,1


In [3]:
df_1 = df.groupby(['user_id_1', 'group'], as_index=False).agg({'is_match': ['sum', 'count']})
df_1.columns = ['user_id_1', 'group', 'is_match_sum', 'is_match_count']

In [4]:
df_1['diff'] = round(df_1['is_match_sum']/df_1['is_match_count'], 2)
df_1

,user_id_1,group,is_match_sum,is_match_count,diff
0,1,1,11,24,0.46
1,2,1,7,16,0.44
2,3,1,5,16,0.31
3,4,0,2,9,0.22
4,5,1,13,22,0.59
...,...,...,...,...,...
995,996,0,1,8,0.12
996,997,0,1,12,0.08
997,998,1,10,18,0.56
998,999,0,2,7,0.29


In [8]:
mean_0 = df_1.query('group==0')['diff'].mean()
mean_1 = df_1.query('group==1')['diff'].mean()

print(f'Средняя доля метчей в группе 0 = {mean_0}')
print(f'Средняя доля метчей в группе 1 = {mean_1}')

Средняя доля метчей в группе 0 = 0.1935928143712575
Средняя доля метчей в группе 1 = 0.4024048096192385


В группе с новым алгоритмом (номер 1) средняя доля метчей значительно выше, надо проверить является ли это отличие стат значимым

In [55]:
# Перед тем как проверять значимость, нужно оценить является ли распределение доли метчей нормальным
import scipy
scipy.stats.normaltest(df_1['diff'])

NormaltestResult(statistic=34.28754713649689, pvalue=3.585533135061675e-08)

P-value очень маленький. Следовательно, можно сделать вывод, что распределение отличается от нормального

Для решения задачи применим т-критерий Уэлча, так как он обладает высокой устойчивостью к нарушениям нормальности и так как обе переменные количественные. Даже если данные скошены, вероятность совершить ошибку первого рода (найти различие там, где его нет) остается очень близкой к заданному уровню значимости (5%). Тест Уэлча сохраняет свою точность гораздо лучше, чем классический t-тест Стьюдента.Этот статистический тест проверяет, есть ли значимая разница между средними значениями двух независимых групп. Также я добавляю поправку на неравенство дисперсий (equal_var=False) для большей надежности. 

In [9]:
import scipy.stats as st

#достаём значения двух групп
control = df_1.query('group==0')['diff']
test = df_1.query('group==1')['diff']
#сам тест
st.ttest_ind(control, test, equal_var=False)

TtestResult(statistic=-26.481431782585016, pvalue=7.890669157070396e-117, df=973.9371185920583)

P-value также очень маленький (<0,05), поэтому можно отклонить нулевую гипотезу и сделать вывод, что значимая разница есть!

Проверим, есть ли различия в среднем количестве действий на пользователя в двух полученных группах.

In [16]:
df_group_0 = df_1.query('group==0')
df_group_1 = df_1.query('group==1')
action_0 = df_group_0['is_match_count'].sum()/df_group_0['user_id_1'].count()
action_1 = df_group_1['is_match_count'].sum()/df_group_1['user_id_1'].count()

a0 = df_group_0['is_match_count']
a1 = df_group_1['is_match_count']

Для сравнения средних значений доли мэтчей между двумя группами мы применим t-критерий Стьюдента. P-value меньше, чем 0.05. Это значит, что мы можем отвергнуть нулевую гипотезу и сделать такой вывод: средние между двумя группами статистически различаются.

In [18]:
st.ttest_ind(a0, a1)

TtestResult(statistic=-51.85383774946492, pvalue=1.8942877064043142e-285, df=998.0)

# Вывод
Результаты A/B-теста показали, что новая система улучшила долю мэтчей и активность клиентов в продукте. Поэтому новую систему поиска анкет стоит внедрить для всех пользователей.